<a href="https://colab.research.google.com/github/chantakorn-uranukul-5792092/chantakorn-uranukul-5792092-AnalyticsEngineering/blob/main/2_03_practice_bike_hire_student_attempt1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=17M9oGaOj4KdKiDDH3oTibxJ9qBTSjp6n)

# 2.03 Practice session
## Bike-hire trips

## How this practice works

This is a **short rehearsal** of the timed test. You do not need to finish or get the "right" answer. The aim is to practise the way of working.

- You have about **45 minutes**. Work individually.
- You may use **any AI tool** (for example Gemini in Colab). We are **not marking** anything today. In the real test, the quality of AI-generated code will not matter. What matters is the control youDshow.
- Write in markdown cells as you go: your thinking, **each prompt you send (paste it as you send it)**, and the checks you make.
- Unfinished work is fine. Say what you did not get to.
- At the end, download the Notebook (using File > Download > Download .ipynb). Once it has completed please zip the file up. This is the artefact you would submit for the in-class test.
- You can also save a copy to your GitHub using the instructions from my.wbs (1b asychronous tasks).

**Suggested timing**

| Minutes | Focus |
| --- | --- |
| 0 to 10 | Look at the tables and note what you find |
| 10 to 25 | Clean and prepare the data |
| 25 to 38 | Join and aggregate to answer the question |
| 38 to 45 | Check your answer and write up |

At the end, use **Runtime, then Restart session and run all** and note anything that fails.


## The data

A fictional bike-hire scheme: three tables covering trips taken in March 2025.

| Table | What it holds |
| --- | --- |
| `trips` | One record per trip: bike, start and end station, start time, duration and user type |
| `bikes` | The bikes, with type and purchase year |
| `stations` | The docking stations, with zone and capacity |

**How they connect:** `trips.bike_id` to `bikes.bike_id`, and `trips.start_station_id` (and `end_station_id`) to `stations.station_id`.

Check the real column names in the next cell before you write anything.


In [ ]:
# ---- Setup: run this cell first ----
!pip install -q duckdb

import glob
import os

import duckdb
import pandas as pd

DATA_URL = "https://github.com/MJMortensonWarwick/adminaeaiv/releases/download/dbsession/bike-hire-practice.zip"

DATA_DIR = "/content/bikes"
# Download only if no CSV files are there yet. The CSVs may sit inside a folder
# within the zip; the search below looks in sub-folders too.
if DATA_URL and not glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True):
    import urllib.request
    import zipfile
    os.makedirs(DATA_DIR, exist_ok=True)
    urllib.request.urlretrieve(DATA_URL, "/content/bikes.zip")
    zipfile.ZipFile("/content/bikes.zip").extractall(DATA_DIR)

FILES = {"trips": "trips.csv", "bikes": "bikes.csv", "stations": "stations.csv"}

tables, missing = {}, []
for name, filename in FILES.items():
    found = glob.glob(os.path.join(DATA_DIR, "**", filename), recursive=True)
    if found:
        tables[name] = pd.read_csv(found[0])
    else:
        missing.append(filename)

if missing:
    found_csvs = sorted(os.path.basename(f) for f in glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True))
    raise FileNotFoundError(
        f"Missing files: {missing}. CSV files found under {DATA_DIR}: {found_csvs}. Ask for help."
    )

# Make every table available as a DuckDB table, so you can use SQL or pandas.
con = duckdb.connect()
for name, df in tables.items():
    con.register(name, df)


def q(sql):
    """Run SQL against the tables above and return a pandas DataFrame."""
    return con.sql(sql).df()


for name, df in tables.items():
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")
    print("   ", list(df.columns))


trips: 612 rows, 7 columns
    ['trip_id', 'bike_id', 'start_station_id', 'end_station_id', 'start_time', 'duration_min', 'user_type']
bikes: 60 rows, 3 columns
    ['bike_id', 'bike_type', 'purchase_year']
stations: 12 rows, 4 columns
    ['station_id', 'station_name', 'zone', 'capacity']


## Examples: two ways to work with the tables

You can use SQL, pandas, or a mix of both. The next two cells show each. Run them and change them to see what happens. You do not have to use them in your answer.


In [ ]:
# Example 1: SQL
# q() runs SQL against any of the tables above and returns a pandas DataFrame.
from IPython.display import display

display(q("SELECT * FROM trips LIMIT 5"))

display(q("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT trip_id) AS n_distinct_trips FROM trips"))

display(q("""
    SELECT t.trip_id, s.station_name
    FROM trips AS t
    JOIN stations AS s ON t.start_station_id = s.station_id
    LIMIT 5
"""))


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
0,T0001,B028,S05,S01,2025-03-01 05:52:00,21.0,Casual
1,T0002,B043,S12,S12,2025-03-01 06:31:00,27.0,CASUAL
2,T0003,B045,S11,S07,2025-03-01 07:06:00,18.0,Casual
3,T0004,B051,S04,S04,2025-03-01 07:06:00,34.0,Member
4,T0005,B049,S01,S09,2025-03-01 07:36:00,16.0,Member


,n_rows,n_distinct_trips
0,612,600


,trip_id,station_name
0,T0001,Park Road
1,T0002,Cathedral Close
2,T0003,Sports Centre
3,T0004,University Gate
4,T0005,Market Square


In [ ]:
# Example 2: pandas
# Each table is also a pandas DataFrame, stored in the dictionary `tables`.
from IPython.display import display

trips = tables["trips"]
display(trips.head())
print(trips.shape)
print(trips.dtypes)

joined = tables["trips"].merge(tables["stations"], left_on="start_station_id", right_on="station_id", how="inner")
display(joined[["trip_id", "station_name"]].head())

# You can mix the two: run SQL, then use pandas methods on the result
result = q("SELECT start_station_id FROM trips")
print(result["start_station_id"].nunique(), "different start stations in trips")


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
0,T0001,B028,S05,S01,2025-03-01 05:52:00,21.0,Casual
1,T0002,B043,S12,S12,2025-03-01 06:31:00,27.0,CASUAL
2,T0003,B045,S11,S07,2025-03-01 07:06:00,18.0,Casual
3,T0004,B051,S04,S04,2025-03-01 07:06:00,34.0,Member
4,T0005,B049,S01,S09,2025-03-01 07:36:00,16.0,Member


(612, 7)
trip_id              object
bike_id              object
start_station_id     object
end_station_id       object
start_time           object
duration_min        float64
user_type            object
dtype: object


,trip_id,station_name
0,T0001,Park Road
1,T0002,Cathedral Close
2,T0003,Sports Centre
3,T0004,University Gate
4,T0005,Market Square


12 different start stations in trips


## Your brief

**From:** the scheme manager

> We want to know whether electric bikes are used for longer trips than standard bikes, and whether that differs by zone. Please give us the **average trip duration by bike type and by the zone of the start station**.

![](https://drive.google.com/uc?export=view&id=1eH5Q3l4LnzwZ-XcT_kJDeHoKA2rufiOW)

**What your final table should look like**

One row per combination of bike type and zone, with the number of trips and the average duration in minutes. State what one row represents.

**Things to think about** (you do not have to solve all of them)

- Are there missing, duplicated or odd values in the tables you use? What will you do about each, and why?
- Do the bike types and dates look consistent?
- How will you check your final table is believable?


## 1. Look at the data

*What does one row of each table represent? Which columns will you use? What looks odd, missing or inconsistent? Write down what you find.*

In [ ]:
#For table bikes, each row represent each bike with type and purchase year
#For table trips, each row represent each trip: bike, start and end station, start time, duration and user type
#For table stations, each row represent each station with zone and capacity

#I will be using column bike_type, zone, duration_min and station_name for the final result. I will be using column bike_id and station_id to connect between table

from IPython.display import display

display(q("SELECT * FROM bikes LIMIT 10"))

display(q("SELECT * FROM trips LIMIT 10"))

display(q("SELECT * FROM stations LIMIT 10"))

#From quick look at the data, there are many missing value for duration_min and capacity in table trips. Also the format for user_type, start_time, bike_type is inconsistent.
#I also ask AI to check whether I found all the inconsistency, here is the response
#The three files (bikes, stations, trips) are structurally sound but need cleaning before analysis. IDs link up across tables, and I found no orphan bikes or stations from reading the data (confirm with the orphan checks in the script).
#Missing values. Two columns have gaps: duration_min in trips (about 30 blanks) and capacity in stations (only S09 is blank). The earlier observation placed capacity in trips, but it belongs to stations.
#Inconsistent formats. Three columns are affected: bike_type (case variants, trailing spaces, and E-bike), user_type (case variants and trailing spaces), and start_time (stored as text in mixed date formats).
#Odd values. The most serious are 12 duplicated trip rows (612 raw rows reduce to 600 trips) and three trips lasting 24 to 48 hours, which are almost certainly errors. Overlapping trips on the same bike, and bikes whose end and start stations don’t connect, look like synthetic-data artefacts. I’d flag these in your write-up rather than delete them.

,bike_id,bike_type,purchase_year
0,B001,Standard,2021
1,B002,Electric,2022
2,B003,electric,2024
3,B004,Standard,2021
4,B005,STANDARD,2022
5,B006,Standard,2024
6,B007,standard,2024
7,B008,Standard,2021
8,B009,Standard,2021
9,B010,Standard,2023


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
0,T0001,B028,S05,S01,2025-03-01 05:52:00,21.0,Casual
1,T0002,B043,S12,S12,2025-03-01 06:31:00,27.0,CASUAL
2,T0003,B045,S11,S07,2025-03-01 07:06:00,18.0,Casual
3,T0004,B051,S04,S04,2025-03-01 07:06:00,34.0,Member
4,T0005,B049,S01,S09,2025-03-01 07:36:00,16.0,Member
5,T0006,B040,S11,S09,2025-03-01 08:06:00,15.0,Member
6,T0007,B044,S12,S01,01/03/2025 09:26,10.0,Member
7,T0008,B010,S06,S12,2025-03-01 10:35:00,6.0,CASUAL
8,T0009,B042,S02,S11,2025-03-01 12:14:00,12.0,Member
9,T0010,B015,S06,S12,2025-03-01 13:04:00,28.0,Casual


,station_id,station_name,zone,capacity
0,S01,Market Square,Centre,29.0
1,S02,Railway Station,Centre,23.0
2,S03,Canal Basin,North,24.0
3,S04,University Gate,West,28.0
4,S05,Park Road,South,22.0
5,S06,Library Corner,Centre,26.0
6,S07,Hospital Entrance,East,27.0
7,S08,Riverside Walk,South,14.0
8,S09,Old Town Hall,Centre,NaN
9,S10,Retail Park,North,16.0


## 2. Clean and prepare

*Decide what to do about each problem you found, say why, and do it. Short cells that you can re-run are better than one long cell.*

In [ ]:
#ask AI to generate the code using pandas to clean the data based on the inconsistent they found
import pandas as pd
import numpy as np

# ============================================================
# 0. Copy raw tables (originals stay untouched in `tables`)
# ============================================================
bikes    = tables["bikes"].copy()
stations = tables["stations"].copy()
trips    = tables["trips"].copy()

# ============================================================
# 1. Strip whitespace in all text columns (object AND string dtypes)
# ============================================================
for df in (bikes, stations, trips):
    for col in df.select_dtypes(include=["object", "string"]).columns:
        df[col] = df[col].astype("string").str.strip()

# ============================================================
# 2. bikes.bike_type -> Standard / Electric
# ============================================================
bikes["bike_type"] = (bikes["bike_type"].str.lower()
                      .replace({"e-bike": "electric", "ebike": "electric"})
                      .str.title())
assert set(bikes["bike_type"].dropna()) <= {"Standard", "Electric"}, \
    f"Unexpected bike types: {bikes['bike_type'].unique()}"
assert bikes["bike_id"].is_unique

# ============================================================
# 3. trips.user_type -> Casual / Member
# ============================================================
trips["user_type"] = trips["user_type"].str.title()
assert set(trips["user_type"].dropna()) <= {"Casual", "Member"}, \
    f"Unexpected user types: {trips['user_type'].unique()}"

# ============================================================
# 4. trips.start_time: parse each pattern explicitly
# ============================================================
s      = trips["start_time"].astype(str).str.strip()
is_iso = s.str.match(r"^\d{4}-\d{1,2}-\d{1,2}")
is_dmy = s.str.match(r"^\d{1,2}/\d{1,2}/\d{4}")
assert (is_iso | is_dmy).all(), f"Unrecognised formats:\n{s[~(is_iso | is_dmy)].head()}"

parsed = pd.Series(pd.NaT, index=trips.index, dtype="datetime64[ns]")

# 4a. Day/month/year rows (with or without seconds)
x = s[is_dmy]
dmy = pd.to_datetime(x, format="%d/%m/%Y %H:%M", errors="coerce")
dmy = dmy.fillna(pd.to_datetime(x, format="%d/%m/%Y %H:%M:%S", errors="coerce"))
parsed.loc[is_dmy] = dmy

# 4b. ISO-style rows. The earlier error hinted these may be year-day-month,
#     so parse both ways and keep whichever matches the month seen in the
#     unambiguous day/month/year rows.
if is_iso.any():
    x = s[is_iso]
    ymd = pd.to_datetime(x, format="ISO8601", errors="coerce")
    ydm = pd.to_datetime(x, format="%Y-%d-%m %H:%M:%S", errors="coerce") \
            .fillna(pd.to_datetime(x, format="%Y-%d-%m %H:%M", errors="coerce"))
    if is_dmy.any():
        ref_month = dmy.dt.month.mode().iloc[0]
        iso_parsed = ymd.where(ymd.dt.month == ref_month,
                               ydm.where(ydm.dt.month == ref_month, ymd))
    else:
        iso_parsed = ymd
    parsed.loc[is_iso] = iso_parsed

trips["start_time"] = parsed
assert trips["start_time"].notna().all(), "Some start_time values failed to parse"

# ============================================================
# 5. Remove duplicate trips (after parsing, so formatting can't hide them)
# ============================================================
n_raw = len(trips)
trips = trips.drop_duplicates(subset="trip_id", keep="first").reset_index(drop=True)
print(f"Removed {n_raw - len(trips)} duplicate rows -> {len(trips)} trips")

# ============================================================
# 6. duration_min: missing + impossible values
# ============================================================
MAX_REASONABLE_MIN = 120                      # judgement call, adjust if needed

trips["duration_raw"]      = trips["duration_min"]
trips["duration_outlier"]  = trips["duration_min"] > MAX_REASONABLE_MIN
trips["duration_missing"]  = trips["duration_min"].isna()
trips["duration_imputed"]  = trips["duration_outlier"] | trips["duration_missing"]

trips.loc[trips["duration_outlier"], "duration_min"] = np.nan

# bike_type via map (avoids merge side effects)
trips["bike_type"] = trips["bike_id"].map(bikes.set_index("bike_id")["bike_type"])

grp_median = trips.groupby(["user_type", "bike_type"])["duration_min"].transform("median")
trips["duration_min"] = (trips["duration_min"]
                         .fillna(grp_median)
                         .fillna(trips["duration_min"].median()))

# ============================================================
# 7. stations.capacity: fill with the zone median, flagged
# ============================================================
stations["capacity_imputed"] = stations["capacity"].isna()
zone_median = stations.groupby("zone")["capacity"].transform("median")
stations["capacity"] = (stations["capacity"]
                        .fillna(zone_median)
                        .fillna(stations["capacity"].median()))

# ============================================================
# 8. Derived columns
# ============================================================
trips["end_time"]   = trips["start_time"] + pd.to_timedelta(trips["duration_min"], unit="m")
trips["round_trip"] = trips["start_station_id"] == trips["end_station_id"]
trips["date"]       = trips["start_time"].dt.date
trips["hour"]       = trips["start_time"].dt.hour

# ============================================================
# 9. Flag logical oddities (flagged, NOT deleted)
# ============================================================
t = trips.sort_values(["bike_id", "start_time"])
prev_end_time    = t.groupby("bike_id")["end_time"].shift()
prev_end_station = t.groupby("bike_id")["end_station_id"].shift()
has_prev         = prev_end_time.notna()

# Same bike starts a new trip before its previous trip has ended
trips["overlaps_prev_trip"] = (t["start_time"] < prev_end_time).reindex(trips.index).fillna(False)

# Bike starts somewhere other than where its previous trip ended
trips["station_mismatch"] = ((t["start_station_id"] != prev_end_station) & has_prev) \
                              .reindex(trips.index).fillna(False)

# Any row you may want to exclude from sensitive analysis
trips["any_quality_flag"] = (trips["duration_imputed"]
                             | trips["overlaps_prev_trip"]
                             | trips["station_mismatch"])

# ============================================================
# 10. Validation (fails loudly if anything is wrong)
# ============================================================
assert set(bikes["bike_type"]) == {"Standard", "Electric"}
assert set(trips["user_type"]) == {"Casual", "Member"}
assert trips["trip_id"].is_unique
assert trips["start_time"].notna().all()
assert (trips["start_time"].dt.year == 2025).all()
assert (trips["start_time"].dt.month == 3).all(), \
    f"Months found: {sorted(trips['start_time'].dt.month.unique())} (day/month swap?)"
assert trips["duration_min"].notna().all()
assert trips["duration_min"].between(1, MAX_REASONABLE_MIN).all()
assert stations["capacity"].notna().all()
assert trips["bike_id"].isin(bikes["bike_id"]).all(),             "orphan bike_ids"
assert trips["start_station_id"].isin(stations["station_id"]).all(), "orphan start stations"
assert trips["end_station_id"].isin(stations["station_id"]).all(),   "orphan end stations"
assert trips["bike_type"].notna().all()

# ============================================================
# 11. Summary
# ============================================================
print("\n--- SUMMARY ---")
print("trips:", len(trips), "| date range:", trips["start_time"].min(), "->", trips["start_time"].max())
print("durations imputed:", trips["duration_imputed"].sum(),
      f"(missing {trips['duration_missing'].sum()}, outliers {trips['duration_outlier'].sum()})")
print("overlapping trips flagged:", trips["overlaps_prev_trip"].sum())
print("station mismatches flagged:", trips["station_mismatch"].sum())
print("capacity imputed:\n", stations.loc[stations["capacity_imputed"],
                                          ["station_id", "station_name", "capacity"]].to_string(index=False))
print("\nbike types:\n", bikes["bike_type"].value_counts().to_string())
print("\nuser types:\n", trips["user_type"].value_counts().to_string())

# ============================================================
# 12. Register cleaned tables for SQL
# ============================================================
con.register("bikes_clean", bikes)
con.register("stations_clean", stations)
con.register("trips_clean", trips)

Removed 12 duplicate rows -> 600 trips

--- SUMMARY ---
trips: 600 | date range: 2025-03-01 05:52:00 -> 2025-03-31 21:54:00
durations imputed: 33 (missing 30, outliers 3)
overlapping trips flagged: 3
station mismatches flagged: 495
capacity imputed:
 station_id  station_name  capacity
       S09 Old Town Hall      27.0

bike types:
 bike_type
Standard    47
Electric    13

user types:
 user_type
Member    386
Casual    214


## 3. Join and aggregate

*Join the tables and calculate the answer. Note how many rows you expect after each join, then check.*

In [ ]:
#ask AI to join the table and calculate answer
result = q("""
    SELECT
        b.bike_type,
        s.zone                          AS start_zone,
        COUNT(*)                        AS trips,
        ROUND(AVG(t.duration_min), 2)   AS avg_duration_min
    FROM trips_clean    AS t
    JOIN bikes_clean    AS b ON t.bike_id          = b.bike_id
    JOIN stations_clean AS s ON t.start_station_id = s.station_id
    GROUP BY b.bike_type, s.zone
    ORDER BY b.bike_type, s.zone
""")
display(result)

,bike_type,start_zone,trips,avg_duration_min
0,Electric,Centre,87,14.40
1,Electric,East,12,15.50
2,Electric,North,24,15.83
3,Electric,South,26,13.38
4,Electric,West,25,15.36
5,Standard,Centre,227,18.82
6,Standard,East,24,19.08
7,Standard,North,39,20.36
8,Standard,South,60,19.43
9,Standard,West,76,19.71


## 4. Check

*How do you know the answer is right? Compare a count, spot check a few records, or recalculate one number a different way. What did you find?*

## 5. Prompts and AI notes

*Paste each prompt as you sent it. Note what you accepted, changed or rejected and why.*

## 6. What I could not verify

*What remains unchecked? What would you do next?*